In [ ]:
import sys,os
hsaDir = os.path.abspath("../../")
srcDir = os.path.join(hsaDir,"src")
sys.path.insert(0,srcDir)

In [ ]:
from heliostat_surface_analysis.loader.scaling_loader import ScalingLoader
from heliostat_surface_analysis.loader.default_loader import DefaultLoader
from heliostat_surface_analysis.preprocessor.basic_preprocessor import BasicPreprocessor
from heliostat_surface_analysis.preprocessor.no_preprocessor import NoPreprocessor
from heliostat_surface_analysis.mesher.delaunay_mesher import DelaunayMesher
from heliostat_surface_analysis.mesher.bspline_mesher import BSplineMesher
from heliostat_surface_analysis.analyser.fit_surface_analyser import FitSurfaceAnalyser
from heliostat_surface_analysis.analyser.fixed_focal_length_analyser import FixedFocalLengthAnalyser
from heliostat_surface_analysis.aggregator.aggregator import WeightedMLEAggregator
from heliostat_surface_analysis.analysis_manager import AnalysisManager
from heliostat_surface_analysis.plotting.plotting import plot_2d_surface,set_plot_font_size

In [ ]:
import pandas as pd
import progressbar
import matplotlib.pyplot as plt
import numpy as np
import random

In [ ]:
%matplotlib inline

In [ ]:
# for progressbar
widgets = [' [',
         progressbar.Timer(format= 'elapsed time: %(elapsed)s'),
         '] ',
           progressbar.Bar('*'),' (',
           progressbar.ETA(), ') ',
          ]

### Setup analysis components

In [ ]:
preprocessor = BasicPreprocessor()
meshers = {}
# NOTE: setting the max_aspect_ratio is important
meshers["Delaunay"] = DelaunayMesher({'max_aspect_ratio':10})
meshers["B-spline"] = BSplineMesher({'polynomial_degree':3,'dx':0.01,'dy':0.01})
analysers = {}
analysers["FixedFocalLength"] = None # placeholder for ordering, will be filled in below
analysers["FitSurface (paraboloid)"] = FitSurfaceAnalyser({"surface":"circular_paraboloid"})
analysers["FitSurface (deformed paraboloid)"] = FitSurfaceAnalyser({"surface":"anu_polynomial"})
aggregator = WeightedMLEAggregator()

In [ ]:
scansCase = "Field2 PEG scans"

In [ ]:
if scansCase=="Field2 PEG scans":
    loader = ScalingLoader({"units":"mm"})
    scansDir = os.path.join(hsaDir,"examples","command_line","circular_paraboloids","Field2_PEG","scans")
    designFocalLengthsFile = os.path.join(hsaDir,"examples","command_line","circular_paraboloids","Field2_PEG","design_focal_lengths.txt")
    analysers["FixedFocalLength"] = FixedFocalLengthAnalyser({"focal_length_file":designFocalLengthsFile})
elif scansCase=="Field1 scans":
    loader = DefaultLoader()
    scansDir = os.path.join(hsaDir,"examples","command_line","circular_paraboloids","Field1_stage4_2023","scans")
    designFocalLengthsFile = os.path.join(hsaDir,"examples","command_line","circular_paraboloids","Field1_stage4_2023","design_focal_lengths.txt")
    analysers["FixedFocalLength"] = FixedFocalLengthAnalyser({"focal_length_file":designFocalLengthsFile})
elif scansCase=="circular paraboloid validation":
    loader = DefaultLoader()
    scansDir = os.path.join(hsaDir,"examples","command_line","circular_paraboloids","synthetic_datasets","scans")
    designFocalLengthsFile = os.path.join(hsaDir,"examples","command_line","circular_paraboloids","synthetic_datasets","design_focal_lengths.txt")
    analysers["FixedFocalLength"] = FixedFocalLengthAnalyser({"focal_length":29.})
else:
    raise RuntimeError("not recognised")

In [ ]:
scanFiles = sorted(os.listdir(scansDir))
print(len(scanFiles))

### Compare meshers

In [ ]:
analyser = analysers["FitSurface (paraboloid)"]
scanNames = []
focalLengths = []
slopeErrors = {}
for mesherName,_ in meshers.items():
    slopeErrors[mesherName] = []
bar = progressbar.ProgressBar(maxval=len(scanFiles)).start()
for i,scanFile in enumerate(scanFiles):
    scanPath = os.path.join(scansDir,scanFile)
    for mesherName,mesher in meshers.items():
        result = AnalysisManager.analyse_scan(scanPath,loader,preprocessor,mesher,analyser,aggregator,output_dir=None,plot=False,write_stls=False)
        slopeErrors[mesherName].append(result["aggregate"])
        focalLength = 1/4./result["coeff: a(x^2 + y^2)"]
        scanName = result["name"]
    # these are the same irrespective of what mesher is used
    scanNames.append(scanName)
    focalLengths.append(focalLength)
    bar.update(i+1)
bar.finish()

In [ ]:
dataDict = {}
dataDict["scan"] = scanNames
dataDict["f"] = focalLengths
for mesherName,mesher in meshers.items():
    dataDict[f"sigma ({mesherName})"] = slopeErrors[mesherName]
df = pd.DataFrame(dataDict)
df.set_index("scan",inplace=True)
df.head(n=150)

### Compare analysers

In [ ]:
mesher = meshers["Delaunay"]
scanNames = []
slopeErrors = {}
focalLengths = {}
for analyserName,_ in analysers.items():
    slopeErrors[analyserName] = []
    focalLengths[analyserName] = []
bar = progressbar.ProgressBar(maxval=len(scanFiles)).start()
for i,scanFile in enumerate(scanFiles):
    scanPath = os.path.join(scansDir,scanFile)
    for analyserName,analyser in analysers.items():
        result = AnalysisManager.analyse_scan(scanPath,loader,preprocessor,mesher,analyser,aggregator,output_dir=None,plot=False,write_stls=False)
        slopeErrors[analyserName].append(result["aggregate"])
        scanName = result["name"]
        if analyserName=="FitSurface (paraboloid)":
            f = 1/4./result["coeff: a(x^2 + y^2)"]
        elif analyserName=="FitSurface (deformed paraboloid)":
            # NOTE: this the geometric average of fx and fy, as a single 'focal length' doesn't exist for the deformed paraboloid
            f = 1/4./np.sqrt(result["coeff: cx^2"]**2 + result["coeff: gy^2"]**2)
        else:
            f = analyser.focal_length_model.get(scanName)
        focalLengths[analyserName].append(f)
    # these are the same irrespective of what analyser is used
    scanNames.append(scanName)
    bar.update(i+1)
bar.finish()

In [ ]:
dataDict = {}
dataDict["scan"] = scanNames
for analyserName,_ in analysers.items():
    dataDict[f"f ({analyserName})"] = focalLengths[analyserName]
    dataDict[f"sigma ({analyserName})"] = slopeErrors[analyserName]
df = pd.DataFrame(dataDict)
df["eps(f)"] = (df["f (FixedFocalLength)"]-df["f (FitSurface (paraboloid))"])/df["f (FitSurface (paraboloid))"]
df.set_index("scan",inplace=True)
with pd.ExcelWriter('facets_analysis.ods', engine="odf") as doc:
    df.to_excel(doc, sheet_name="Sheet1")
df.head(n=150)

In [ ]:
for analyserName,_ in analysers.items():
    print("%s, mean slope error = %f mrad"%(analyserName,df["sigma (%s)"%analyserName].mean()*1e3))

### plotting of slope error distribution for the scanned facet population

In [ ]:
# NOTE: may want to filter the population before plotting, e.g. removing scans with abnormally high slope error
sigma_max = 10.0e-3
df_plotting = df.drop(df[df["sigma (FitSurface (deformed paraboloid))"]>sigma_max].index)
print("%d out of %d scans retained for plotting"%(len(df_plotting),len(df)))

In [ ]:
f,ax = plt.subplots(3,1)
f.set_size_inches(14,18)
for i,analyserName in enumerate(analysers.keys()):
    ax[i].set_title(analyserName)
    ax[i].hist(df_plotting["sigma (%s)"%analyserName]*1e3,bins=20,rwidth=0.85)
    min_ylim, max_ylim = ax[i].get_ylim()
    mean = df_plotting["sigma (%s)"%analyserName].mean()*1e3
    ax[i].axvline(mean, color='red', linestyle='dashed', linewidth=3)
    ax[i].text(mean*1.1, max_ylim*0.9, '$\mu$ = {:.2f} mrad'.format(mean),color='red',size=16)
    ax[i].grid()
    if i==len(analysers)-1:
        ax[i].set_xlabel("slope error (mrad)")
    ax[i].set_ylabel("Frequency (counts per bin)")
    #ax[i].set_xlim(0,3.6)
    set_plot_font_size(ax[i],16,False)

### plotting of focal length error distribution for the scanned facet population

In [ ]:
f,ax = plt.subplots()
f.set_size_inches(14,6)
ax.grid()
ax.hist(df_plotting["eps(f)"]*100,bins=50,rwidth=0.85)
ax.set_xlabel("Focal length error (%)")
ax.set_ylabel("Frequency (counts per bin)")
set_plot_font_size(ax,18,False)
plt.tight_layout()
plt.savefig("field_focal_length_error_histogram.png")